<a href="https://colab.research.google.com/github/khalidkhankakar/Hands-on-Machine-Learning/blob/master/chapter_10_ANN/neural_network_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Linear Reqression with Pytorch

In [1]:
import torch
from sklearn.datasets import fetch_california_housing
X, y = fetch_california_housing(return_X_y=True)
X.shape, y.shape


((20640, 8), (20640,))

In [2]:
from sklearn.model_selection import train_test_split
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, random_state=42)
X_train, X_dev, y_train, y_dev = train_test_split(X_train_full, y_train_full, random_state=42)
X_train.shape, y_train.shape, X_test.shape, y_test.shape, X_dev.shape, y_dev.shape

((11610, 8), (11610,), (5160, 8), (5160,), (3870, 8), (3870,))

In [3]:
X_train = torch.FloatTensor(X_train)
X_dev = torch.FloatTensor(X_dev)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdim=True)
stds = X_train.std(dim=0, keepdim=True)
X_train = (X_train - means) / stds
X_test = (X_test- means) / stds
X_dev = (X_dev - means) / stds

In [4]:
y_train= torch.FloatTensor(y_train.reshape(-1, 1))
y_dev= torch.FloatTensor(y_dev.reshape(-1, 1))
y_test= torch.FloatTensor(y_test.reshape(-1, 1))

In [5]:
# Our parameter for network
n_features = X_train.shape[1]
weights = torch.randn((n_features, 1), requires_grad=True)
bias = torch.tensor(0., requires_grad=True)

In [6]:
# let's train the model
learning_rate = 0.01
n_epochs = 20

for epoch in range(n_epochs):
  y_pred = X_train @ weights + bias
  loss = ((y_pred - y_train)**2).mean()
  loss.backward()
  with torch.no_grad():
    bias -= learning_rate * bias.grad
    weights -= learning_rate * weights.grad
    bias.grad.zero_()
    weights.grad.zero_()
  print(f"Epoch {epoch + 1}/{n_epochs}, Loss {loss.item()}")

Epoch 1/20, Loss 10.622960090637207
Epoch 2/20, Loss 10.217225074768066
Epoch 3/20, Loss 9.828619003295898
Epoch 4/20, Loss 9.456372261047363
Epoch 5/20, Loss 9.099754333496094
Epoch 6/20, Loss 8.758069038391113
Epoch 7/20, Loss 8.43065357208252
Epoch 8/20, Loss 8.116878509521484
Epoch 9/20, Loss 7.816143035888672
Epoch 10/20, Loss 7.52787446975708
Epoch 11/20, Loss 7.251531600952148
Epoch 12/20, Loss 6.986591339111328
Epoch 13/20, Loss 6.732562065124512
Epoch 14/20, Loss 6.488968849182129
Epoch 15/20, Loss 6.255364894866943
Epoch 16/20, Loss 6.0313191413879395
Epoch 17/20, Loss 5.816422939300537
Epoch 18/20, Loss 5.610286235809326
Epoch 19/20, Loss 5.412534713745117
Epoch 20/20, Loss 5.222813129425049


In [7]:
X_new = X_test[:3]
with torch.no_grad():
  y_pred = X_new @ weights + bias

In [8]:
y_pred

tensor([[ 0.8578],
        [ 0.9258],
        [-0.3362]])

Linear Regression with torch high level API

In [9]:
import torch.nn as nn
model = nn.Linear(in_features= n_features, out_features = 1)

In [10]:
# parameters
model.bias, model.weight

(Parameter containing:
 tensor([-0.1648], requires_grad=True),
 Parameter containing:
 tensor([[-0.1345, -0.0587, -0.1247, -0.1204,  0.1577,  0.1509,  0.3228, -0.0119]],
        requires_grad=True))

In [11]:
for param in model.parameters():
  print(param)
for named_param in model.named_parameters():
  print(named_param[0], named_param[1])

Parameter containing:
tensor([[-0.1345, -0.0587, -0.1247, -0.1204,  0.1577,  0.1509,  0.3228, -0.0119]],
       requires_grad=True)
Parameter containing:
tensor([-0.1648], requires_grad=True)
weight Parameter containing:
tensor([[-0.1345, -0.0587, -0.1247, -0.1204,  0.1577,  0.1509,  0.3228, -0.0119]],
       requires_grad=True)
bias Parameter containing:
tensor([-0.1648], requires_grad=True)


In [12]:
# here model is just normal.
# so model is not train yet. it will make terriable predications
# Note: when call the this torch will internall call forward() method Which is just as see in previous example. X @ self.weight + self.bias
model(X_train[:3])

tensor([[ 0.2741],
        [-0.1633],
        [ 0.0436]], grad_fn=<AddmmBackward0>)

In [13]:
# optimizer: In short it updates the learnable parameters values like weights and bias. You can choose different kinds algorithm to perform backpropagation
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [14]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
  for epoch in range(n_epochs):
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f'{epoch}/{n_epochs}, loss: {loss.item()}')

In [15]:
train_bgd(model, optimizer, mse, X_train, y_train, 20)

0/20, loss: 7.020485877990723
1/20, loss: 6.759973049163818
2/20, loss: 6.510476112365723
3/20, loss: 6.271505355834961
4/20, loss: 6.042599201202393
5/20, loss: 5.823311805725098
6/20, loss: 5.613221168518066
7/20, loss: 5.411925792694092
8/20, loss: 5.219040870666504
9/20, loss: 5.034200191497803
10/20, loss: 4.8570556640625
11/20, loss: 4.687273025512695
12/20, loss: 4.524535179138184
13/20, loss: 4.368537902832031
14/20, loss: 4.218991756439209
15/20, loss: 4.075620651245117
16/20, loss: 3.938159465789795
17/20, loss: 3.8063576221466064
18/20, loss: 3.6799731254577637
19/20, loss: 3.5587756633758545


Implement MLP with low level API

In [16]:
# input layer weights and bias
w1 = torch.randn((n_features, 64), requires_grad=True)
b1 = torch.zeros(64, requires_grad=True)

# second layer weights and bias
w2 = torch.randn((64, 32), requires_grad=True)
b2 = torch.zeros(32, requires_grad=True)

# Output layer weights and bias
w3 = torch.randn((32, 1), requires_grad=True)
b3 = torch.zeros(1, requires_grad=True)

parameters = [w1, b1, w2, b2, w3, b3]
print(sum(p.nelement() for p in parameters))

2689


In [17]:
n_epochs = 20
for epoch in range(n_epochs):
  # forward pass
  x = X_train @ w1 + b1
  x = torch.relu(x)
  x = x @ w2 + b2
  x = torch.relu(x)
  y_pred = x @ w3 + b3
  loss = ((y_pred - y_train)**2).mean()


  # backward pass
  for p in parameters:
    p.grad= None
  loss.backward()

  # update
  lr = 0.001
  with torch.no_grad():
    for p in parameters:
      p.data -= lr * p.grad

  print(f'{epoch}/{n_epochs}, loss: {loss.item():.4f}')


0/20, loss: 1158.7129
1/20, loss: 1220.7452
2/20, loss: 3445.6431
3/20, loss: 5402.5034
4/20, loss: 340.7776
5/20, loss: 83.7639
6/20, loss: 42.0652
7/20, loss: 29.7785
8/20, loss: 24.8683
9/20, loss: 22.2184
10/20, loss: 20.4167
11/20, loss: 19.0282
12/20, loss: 17.8844
13/20, loss: 16.8994
14/20, loss: 16.0418
15/20, loss: 15.2863
16/20, loss: 14.6124
17/20, loss: 14.0081
18/20, loss: 13.4600
19/20, loss: 12.9624


In [18]:
# predications
with torch.no_grad():
  x = X_dev[:10] @ w1 + b1
  x = torch.relu(x)
  x = x @ w2 + b2
  x = torch.relu(x)
  y_pred = x @ w3 + b3

print(y_pred)


tensor([[-0.2332],
        [ 1.5569],
        [-0.5670],
        [ 1.8277],
        [ 2.0618],
        [-2.8809],
        [ 7.0623],
        [ 4.2416],
        [ 2.5013],
        [ 4.6316]])


MLP with high level API

In [19]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_bgd(model, optimizer, mse, X_train, y_train, 20)

0/20, loss: 5.045480251312256
1/20, loss: 2.0523123741149902
2/20, loss: 1.0039883852005005
3/20, loss: 0.8570139408111572
4/20, loss: 0.7740675210952759
5/20, loss: 0.7225847244262695
6/20, loss: 0.6893726587295532
7/20, loss: 0.6669032573699951
8/20, loss: 0.6507738828659058
9/20, loss: 0.6383934020996094
10/20, loss: 0.6281993389129639
11/20, loss: 0.6193399429321289
12/20, loss: 0.6113173365592957
13/20, loss: 0.6038705706596375
14/20, loss: 0.5968307852745056
15/20, loss: 0.5901119112968445
16/20, loss: 0.5836468935012817
17/20, loss: 0.5774063467979431
18/20, loss: 0.5713554620742798
19/20, loss: 0.565444827079773


In [20]:
# DataLoader: it load the batches of data of the deirsed size and suffle them effectively
from torch.utils.data import DataLoader, TensorDataset
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [21]:
# hardware acceleration
if torch.cuda.is_available():
 device = "cuda"
elif torch.backends.mps.is_available():
 device = "mps"
else:
 device = "cpu"
device

'cuda'

In [22]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

model = model.to(device)

In [23]:
def train(model, optimizer, criterion, train_loader, n_epochs):
  model.train()
  for epoch in range(n_epochs):
    total_loss = 0.
    for X_batch, y_batch in train_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred= model(X_batch)
      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

    mean_loss = total_loss / len(train_loader)
    print(f'{epoch + 1}/ {n_epochs}, loss: {mean_loss:.4f}')


In [24]:
train(model, optimizer, mse, train_loader, 20)

1/ 20, loss: 5.0460
2/ 20, loss: 5.0443
3/ 20, loss: 5.0456
4/ 20, loss: 5.0457
5/ 20, loss: 5.0450
6/ 20, loss: 5.0452
7/ 20, loss: 5.0449
8/ 20, loss: 5.0449
9/ 20, loss: 5.0459
10/ 20, loss: 5.0460
11/ 20, loss: 5.0456
12/ 20, loss: 5.0449
13/ 20, loss: 5.0461
14/ 20, loss: 5.0452
15/ 20, loss: 5.0456
16/ 20, loss: 5.0454
17/ 20, loss: 5.0455
18/ 20, loss: 5.0460
19/ 20, loss: 5.0451
20/ 20, loss: 5.0456


In [25]:
def evaluate(model, data_loader,metric_fn, aggergate_fn=torch.mean ):
  model.eval()
  metrics = []
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch= X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      metric = metric_fn(y_pred, y_batch)
      metrics.append(metric)
  return aggergate_fn(torch.stack(metrics))

In [26]:
dev_dataset = TensorDataset(X_dev, y_dev)
dev_loader = DataLoader(dev_dataset, batch_size=32)
dev_mse = evaluate(model, dev_loader, mse)
dev_mse

tensor(4.8635, device='cuda:0')

In [27]:
def rmse(y_pred, y_true):
  return ((y_pred -y_true).pow(2)).mean().sqrt()
evaluate(model, dev_loader,rmse)

tensor(2.1945, device='cuda:0')

In [28]:
%pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 35.1 MB/s eta 0:00:00


torchmetrics

In [29]:
import torchmetrics
def evaluate(model, data_loader,metric ):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch= X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      metric.update(y_pred, y_batch)

  return metric.compute()

In [30]:
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
evaluate(model, dev_loader, rmse)

tensor(2.2054, device='cuda:0')

Building Nonsequential Model Using Custom Modules


In [31]:
# let's wide and deep model
class WideAndDeepV1(nn.Module):
  def __init__(self, n_features):
    super().__init__()
    self.deep_stack = nn.Sequential(
        nn.Linear(n_features, 50), nn.ReLU(),
        nn.Linear(50, 40), nn.ReLU()
    )
    self.output_layer = nn.Linear(40 + n_features, 1)

  def forward(self, X):
    deep_output = self.deep_stack(X)
    wide_and_deep = torch.concat([X, deep_output], dim=1)
    return self.output_layer(wide_and_deep)


In [32]:
torch.manual_seed(42)
model = WideAndDeepV1(n_features).to(device)
train(model, optimizer, mse,train_loader,  20)

1/ 20, loss: 5.7924
2/ 20, loss: 5.7924
3/ 20, loss: 5.7916
4/ 20, loss: 5.7921
5/ 20, loss: 5.7915
6/ 20, loss: 5.7914
7/ 20, loss: 5.7926
8/ 20, loss: 5.7928
9/ 20, loss: 5.7924
10/ 20, loss: 5.7915
11/ 20, loss: 5.7931
12/ 20, loss: 5.7920
13/ 20, loss: 5.7924
14/ 20, loss: 5.7921
15/ 20, loss: 5.7923
16/ 20, loss: 5.7929
17/ 20, loss: 5.7917
18/ 20, loss: 5.7923
19/ 20, loss: 5.7910
20/ 20, loss: 5.7910


In [33]:
evaluate(model,dev_loader, rmse)

tensor(2.3629, device='cuda:0')

In [34]:
# let's wide and deep model
class WideAndDeepV2(nn.Module):
  def __init__(self, n_features):
    super().__init__()
    self.deep_stack = nn.Sequential(
        nn.Linear(n_features-2, 50), nn.ReLU(),
        nn.Linear(50, 40), nn.ReLU()
    )
    self.output_layer = nn.Linear(40 + 5, 1)

  def forward(self, X):
    X_wide = X[:, :5]
    X_deep = X[:, 2:]
    deep_output = self.deep_stack(X_deep)
    wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
    return self.output_layer(wide_and_deep)


In [35]:
torch.manual_seed(42)
model = WideAndDeepV2(n_features).to(device)
train(model, optimizer, mse,train_loader,  20)

1/ 20, loss: 5.7440
2/ 20, loss: 5.7453
3/ 20, loss: 5.7447
4/ 20, loss: 5.7441
5/ 20, loss: 5.7443
6/ 20, loss: 5.7449
7/ 20, loss: 5.7450
8/ 20, loss: 5.7445
9/ 20, loss: 5.7451
10/ 20, loss: 5.7449
11/ 20, loss: 5.7444
12/ 20, loss: 5.7436
13/ 20, loss: 5.7461
14/ 20, loss: 5.7442
15/ 20, loss: 5.7445
16/ 20, loss: 5.7446
17/ 20, loss: 5.7449
18/ 20, loss: 5.7453
19/ 20, loss: 5.7449
20/ 20, loss: 5.7450


In [36]:
evaluate(model,dev_loader, rmse)

tensor(2.3386, device='cuda:0')

In [51]:
class WideAndDeepV3(nn.Module):
  def __init__(self, n_features):
    super().__init__()
    self.deep_stack = nn.Sequential(
        nn.Linear(n_features-2, 50), nn.ReLU(),
        nn.Linear(50, 40), nn.ReLU(),
        nn.Linear(40, 30), nn.ReLU()
    )
    self.output_layer = nn.Linear(30 + 5, 1)

  def forward(self, X_wide, X_deep):
    deep_output = self.deep_stack(X_deep)
    wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
    return self.output_layer(wide_and_deep)

In [52]:
train_data_wd = TensorDataset(X_train[:, :5], X_train[:, 2:], y_train)
train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True)
X_dev_wd = TensorDataset(X_dev[:, :5], X_dev[:, 2:], y_dev)
dev_loader_wd= DataLoader(X_dev_wd, batch_size=32, shuffle=True)
X_test_wd = TensorDataset(X_test[:, :5], X_test[:, 2:], y_dev)
test_loader_wd= DataLoader(X_test_wd, batch_size=32, shuffle=True)

In [53]:
def evaluate_multi_wd(model, data_loader, metric):
  model.eval()
  metric.reset()

  for X_batch_wide, X_batch_deep, y_batch in data_loader:
    X_batch_wide = X_batch_wide.to(device)
    X_batch_deep = X_batch_deep.to(device)
    y_batch = y_batch.to(device)

    y_pred = model(X_batch_wide, X_batch_deep)
    metric.update(y_pred, y_batch)

  return metric.compute()

def train_multi_wd(model, optimizer, criterion, metric, train_loader, dev_loader, n_epochs):
  history = {"train_losses": [], "train_metrics": [], "dev_metrics": []}

  for epoch in range(n_epochs):
    total_loss = 0
    metric.reset()
    for X_batch_wide, X_batch_deep, y_batch in train_loader:
      model.train()
      X_batch_wide = X_batch_wide.to(device)
      X_batch_deep = X_batch_deep.to(device)
      y_batch = y_batch.to(device)
      y_pred = model(X_batch_wide, X_batch_deep)
      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      metric.update(y_pred, y_batch)
    mean_loss = total_loss / len(train_loader)
    history['train_losses'].append(mean_loss)
    history['dev_metrics'].append(
        evaluate_multi_wd(model, dev_loader, metric).item()
    )
    history['train_metrics'].append(metric.compute().item())

    print(f'{epoch} / {n_epochs}',
          f"train_loss {history['train_losses'][-1]:.4f}",
          f"train_metric {history['train_metrics'][-1]:.4f}",
          f"valid_metric {history['dev_metrics'][-1]:.4f}",
          )
  return history

torch.manual_seed(42)

model = WideAndDeepV3(n_features).to(device)
lr = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0)
mse = nn.MSELoss()
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
history = train_multi_wd(model, optimizer, mse, rmse, train_loader_wd, dev_loader_wd, 20 )

0 / 20 train_loss 0.8366 train_metric 0.6892 valid_metric 0.6892
1 / 20 train_loss 0.4657 train_metric 0.6478 valid_metric 0.6478
2 / 20 train_loss 0.4312 train_metric 0.6474 valid_metric 0.6474
3 / 20 train_loss 0.4215 train_metric 0.6339 valid_metric 0.6339
4 / 20 train_loss 0.4110 train_metric 0.6388 valid_metric 0.6388
5 / 20 train_loss 0.4043 train_metric 0.6320 valid_metric 0.6320
6 / 20 train_loss 0.4004 train_metric 0.7613 valid_metric 0.7613
7 / 20 train_loss 0.4021 train_metric 0.6065 valid_metric 0.6065
8 / 20 train_loss 0.3883 train_metric 0.6023 valid_metric 0.6023
9 / 20 train_loss 0.3797 train_metric 0.6112 valid_metric 0.6112
10 / 20 train_loss 0.3744 train_metric 0.5965 valid_metric 0.5965
11 / 20 train_loss 0.3714 train_metric 0.5903 valid_metric 0.5903
12 / 20 train_loss 0.3696 train_metric 0.5808 valid_metric 0.5808
13 / 20 train_loss 0.3602 train_metric 0.5768 valid_metric 0.5768
14 / 20 train_loss 0.3560 train_metric 0.5851 valid_metric 0.5851
15 / 20 train_loss 0

In [58]:
# build the multi-output model

class DeepAndWideV4(nn.Module):
  def __init__(self, n_features):
    super().__init__()
    self.deep_stack = nn.Sequential(
        nn.Linear(n_features - 2, 50), nn.ReLU(),
        nn.Linear(50, 40), nn.ReLU(),
        nn.Linear(40, 30), nn.ReLU()
    )

    self.output_layer = nn.Linear(30 + 5, 1)
    self.aux_output = nn.Linear(30, 1)

  def forward(self, X_wide, X_deep):
    deep_output = self.deep_stack(X_deep)
    wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
    main_output = self.output_layer(wide_and_deep)
    aux_output = self.aux_output(deep_output)
    return main_output, aux_output




In [59]:
def evaluate_multi_wd_output(model, data_loader, metric):
  model.eval()
  metric.reset()

  for X_batch_wide, X_batch_deep, y_batch in data_loader:
    X_batch_wide = X_batch_wide.to(device)
    X_batch_deep = X_batch_deep.to(device)
    y_batch = y_batch.to(device)

    y_pred, _ = model(X_batch_wide, X_batch_deep)
    metric.update(y_pred, y_batch)
  return metric.compute()

def train_multi_wd_output(model, optimizer, criterion, metric, train_loader, dev_loader, n_epochs):
  history = {"train_losses": [], "train_metrics": [], "dev_metrics": []}

  for epoch in range(n_epochs):
    total_loss = 0
    metric.reset()
    for X_batch_wide, X_batch_deep, y_batch in train_loader:
      model.train()
      X_batch_wide = X_batch_wide.to(device)
      X_batch_deep = X_batch_deep.to(device)
      y_batch = y_batch.to(device)
      y_pred, y_pred_aux = model(X_batch_wide, X_batch_deep)
      main_loss = criterion(y_pred, y_batch)
      aux_loss = criterion(y_pred_aux, y_batch)
      loss = 0.8 * main_loss + 0.2 * aux_loss
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      metric.update(y_pred, y_batch)
    mean_loss = total_loss / len(train_loader)
    history['train_losses'].append(mean_loss)
    history['dev_metrics'].append(
        evaluate_multi_wd_output(model, dev_loader, metric).item()
    )
    history['train_metrics'].append(metric.compute().item())

    print(f'{epoch} / {n_epochs}',
          f"train_loss {history['train_losses'][-1]:.4f}",
          f"train_metric {history['train_metrics'][-1]:.4f}",
          f"valid_metric {history['dev_metrics'][-1]:.4f}",
          )
  return history

torch.manual_seed(42)

model = DeepAndWideV4(n_features).to(device)
lr = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0)
mse = nn.MSELoss()
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
history = train_multi_wd_output(model, optimizer, mse, rmse, train_loader_wd, dev_loader_wd, 20 )

0 / 20 train_loss 1.0693 train_metric 0.7085 valid_metric 0.7085
1 / 20 train_loss 0.5709 train_metric 0.6478 valid_metric 0.6478
2 / 20 train_loss 0.4976 train_metric 0.6451 valid_metric 0.6451
3 / 20 train_loss 0.4688 train_metric 0.6427 valid_metric 0.6427
4 / 20 train_loss 0.4480 train_metric 0.6516 valid_metric 0.6516
5 / 20 train_loss 0.4393 train_metric 0.6161 valid_metric 0.6161
6 / 20 train_loss 0.4246 train_metric 0.6549 valid_metric 0.6549
7 / 20 train_loss 0.4247 train_metric 0.6503 valid_metric 0.6503
8 / 20 train_loss 0.4176 train_metric 0.5981 valid_metric 0.5981
9 / 20 train_loss 0.4030 train_metric 0.6008 valid_metric 0.6008
10 / 20 train_loss 0.3976 train_metric 0.5908 valid_metric 0.5908
11 / 20 train_loss 0.3894 train_metric 0.5843 valid_metric 0.5843
12 / 20 train_loss 0.3868 train_metric 0.6007 valid_metric 0.6007
13 / 20 train_loss 0.3784 train_metric 0.5803 valid_metric 0.5803
14 / 20 train_loss 0.3748 train_metric 0.5940 valid_metric 0.5940
15 / 20 train_loss 0